# Colab Standalone: KuaiRand ALS Gold + Baseline Training

This notebook does **not** clone GitHub and does **not** import project modules. It is self-contained for Colab.

It reads Silver interactions from Google Drive, builds the ALS Gold dataset, trains/evaluates:

1. Popularity baseline
2. Spark implicit ALS baseline

It saves Gold artifacts/factors back to Drive. MLflow logging is optional; if MLflow is not installed, the notebook still writes JSON metrics.

GPU is not required. Use a CPU High-RAM runtime if available.


## Expected Drive Path

Upload this local folder:

```text
data/silver/kuairand/interactions/
```

To this Google Drive path:

```text
MyDrive/recsys/data/silver/kuairand/interactions/
```

The notebook writes outputs to:

```text
MyDrive/recsys/data/gold/als/v1_colab/
MyDrive/recsys/data/mlruns/
```


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
%%bash
set -e
pip install -q pyspark==3.5.6 py4j==0.10.9.7


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
DRIVE_INTERACTIONS = DRIVE_ROOT / 'data/silver/kuairand/interactions'
DRIVE_GOLD_DIR = DRIVE_ROOT / 'data/gold/als/v1_colab'
DRIVE_MLRUNS = DRIVE_ROOT / 'data/mlruns'

LOCAL_ROOT = Path('/content/recsys_work')
LOCAL_SILVER_DIR = LOCAL_ROOT / 'data/silver/kuairand'
LOCAL_INTERACTIONS = LOCAL_SILVER_DIR / 'interactions'
LOCAL_GOLD_DIR = LOCAL_ROOT / 'data/gold/als/v1_colab'
LOCAL_MLRUNS = LOCAL_ROOT / 'data/mlruns'
SPARK_TMP = LOCAL_ROOT / 'spark-tmp'

print('Drive input:', DRIVE_INTERACTIONS)
print('Drive gold output:', DRIVE_GOLD_DIR)
print('Drive MLflow output:', DRIVE_MLRUNS)

if not DRIVE_INTERACTIONS.exists():
    raise FileNotFoundError(f'Missing Drive input: {DRIVE_INTERACTIONS}')


## Copy Data From Drive to Colab Local Disk

Spark is much faster on `/content` than directly on mounted Drive. This cell copies only the required Silver interaction Parquet folder.


In [ ]:
%%bash
set -e
mkdir -p /content/recsys_work/data/silver/kuairand
rm -rf /content/recsys_work/data/silver/kuairand/interactions
cp -r /content/drive/MyDrive/recsys/data/silver/kuairand/interactions /content/recsys_work/data/silver/kuairand/interactions

du -sh /content/recsys_work/data/silver/kuairand/interactions


## Spark, Helpers, and Evaluation Functions

All logic needed for Gold creation, popularity baseline, ALS training, and ranking evaluation is defined below.


In [ ]:
import itertools
import json
import math
import os
import random
import shutil
import subprocess
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except Exception as exc:
    MLFLOW_AVAILABLE = False
    print(f'MLflow is not available; continuing with JSON-only tracking: {exc}')

    class _DummyRunInfo:
        run_id = 'json_only'

    class _DummyRun:
        info = _DummyRunInfo()
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc, tb):
            return False

    class _DummyMlflow:
        def set_tracking_uri(self, *args, **kwargs): pass
        def set_experiment(self, *args, **kwargs): pass
        def start_run(self, *args, **kwargs): return _DummyRun()
        def log_params(self, *args, **kwargs): pass
        def log_param(self, *args, **kwargs): pass
        def log_metric(self, *args, **kwargs): pass
        def log_artifact(self, *args, **kwargs): pass
        def get_tracking_uri(self): return None

    mlflow = _DummyMlflow()

from pyspark.ml.recommendation import ALS
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import SparkSession

os.environ['SPARK_LOCAL_DIRS'] = str(SPARK_TMP)
os.environ['LOCAL_DIRS'] = str(SPARK_TMP)
SPARK_TMP.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('kuairand-colab-als')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.driver.memory', '10g')
    .config('spark.local.dir', str(SPARK_TMP))
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

DEFAULT_WEIGHTS = {
    'watch_ratio': 0.5,
    'long_view': 1.0,
    'is_like': 1.5,
    'is_comment': 1.5,
    'is_forward': 1.5,
    'is_follow': 2.0,
}
FORMULA_VERSION = 'implicit_strength_v1'

@dataclass(frozen=True)
class AlsParams:
    rank: int = 32
    reg_param: float = 0.05
    alpha: float = 10.0
    max_iter: int = 5

def write_json(data, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n')

def write_parquet(df, path: Path, overwrite=True):
    df.write.mode('overwrite' if overwrite else 'errorifexists').parquet(str(path))

def read_interactions(path: Path):
    df = spark.read.parquet(str(path))
    required = {'user_id','video_id','event_ts','time_ms','watch_ratio','long_view','is_like','is_comment','is_forward','is_follow'}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    return df

def split_events(events, train_q=0.70, validation_q=0.85):
    cut_train, cut_validation = events.approxQuantile('time_ms', [train_q, validation_q], 0.001)
    split = (
        events.where(F.col('event_ts').isNotNull())
        .withColumn(
            'split',
            F.when(F.col('time_ms') < F.lit(cut_train), F.lit('train'))
             .when(F.col('time_ms') < F.lit(cut_validation), F.lit('validation'))
             .otherwise(F.lit('test'))
        )
    )
    meta = {
        'train_cutoff_time_ms': int(cut_train),
        'validation_cutoff_time_ms': int(cut_validation),
        'train_cutoff_ts': split.where(F.col('time_ms') >= cut_train).agg(F.min('event_ts')).first()[0].isoformat(),
        'validation_cutoff_ts': split.where(F.col('time_ms') >= cut_validation).agg(F.min('event_ts')).first()[0].isoformat(),
    }
    return split, meta

def add_event_strength(events, weights):
    clipped_watch_ratio = F.least(F.greatest(F.coalesce(F.col('watch_ratio'), F.lit(0.0)), F.lit(0.0)), F.lit(3.0))
    strength = F.lit(weights['watch_ratio']) * clipped_watch_ratio
    for col in ['long_view', 'is_like', 'is_comment', 'is_forward', 'is_follow']:
        strength = strength + F.lit(weights[col]) * F.coalesce(F.col(col), F.lit(0)).cast('double')
    return events.withColumn('event_strength', F.greatest(strength, F.lit(0.0)))

def strong_relevance_expr():
    return (
        (F.coalesce(F.col('long_view'), F.lit(0)) == 1)
        | (F.coalesce(F.col('is_like'), F.lit(0)) == 1)
        | (F.coalesce(F.col('is_comment'), F.lit(0)) == 1)
        | (F.coalesce(F.col('is_forward'), F.lit(0)) == 1)
        | (F.coalesce(F.col('is_follow'), F.lit(0)) == 1)
    )

def graded_relevance_expr(weights):
    return (
        F.lit(weights['long_view']) * F.coalesce(F.col('long_view'), F.lit(0)).cast('double')
        + F.lit(weights['is_like']) * F.coalesce(F.col('is_like'), F.lit(0)).cast('double')
        + F.lit(weights['is_comment']) * F.coalesce(F.col('is_comment'), F.lit(0)).cast('double')
        + F.lit(weights['is_forward']) * F.coalesce(F.col('is_forward'), F.lit(0)).cast('double')
        + F.lit(weights['is_follow']) * F.coalesce(F.col('is_follow'), F.lit(0)).cast('double')
    )

def split_summary(events):
    rows = (
        events.groupBy('split')
        .agg(F.count('*').alias('events'), F.countDistinct('user_id').alias('users'), F.countDistinct('video_id').alias('items'), F.min('event_ts').alias('min_ts'), F.max('event_ts').alias('max_ts'))
        .orderBy('split')
        .collect()
    )
    return [{
        'split': r['split'],
        'events': int(r['events']),
        'users': int(r['users']),
        'items': int(r['items']),
        'min_ts': r['min_ts'].isoformat() if r['min_ts'] else None,
        'max_ts': r['max_ts'].isoformat() if r['max_ts'] else None,
    } for r in rows]

def build_train_interactions(train_events):
    return (
        train_events.groupBy('user_id', 'video_id')
        .agg(F.log1p(F.sum('event_strength')).alias('interaction_strength'))
        .where(F.col('interaction_strength') > 0)
    )

def build_mappings(train_interactions):
    user_window = Window.orderBy('user_id')
    item_window = Window.orderBy('video_id')
    users = train_interactions.select('user_id').distinct().withColumn('user_idx', F.row_number().over(user_window) - 1)
    items = train_interactions.select('video_id').distinct().withColumn('video_idx', F.row_number().over(item_window) - 1)
    return users, items

def apply_mappings(interactions, users, items):
    return (
        interactions.join(F.broadcast(users), 'user_id', 'inner')
        .join(items, 'video_id', 'inner')
        .select('user_id', 'video_id', 'user_idx', 'video_idx', 'interaction_strength')
    )

def build_relevance(events, users, items, split, weights):
    relevant = (
        events.where(F.col('split') == split)
        .where(strong_relevance_expr())
        .withColumn('graded_relevance_event', graded_relevance_expr(weights))
        .groupBy('user_id', 'video_id')
        .agg(F.lit(1).alias('relevant'), F.max('graded_relevance_event').alias('graded_relevance'), F.count('*').alias('relevant_events'))
    )
    return (
        relevant.join(F.broadcast(users), 'user_id', 'left')
        .join(items, 'video_id', 'left')
        .withColumn('is_warm_user', F.col('user_idx').isNotNull())
        .withColumn('is_warm_item', F.col('video_idx').isNotNull())
    )

def cold_start_report(relevance):
    total = relevance.count()
    if total == 0:
        return {'rows': 0, 'warm_user_rate': 0.0, 'warm_item_rate': 0.0, 'cold_user_rate': 0.0, 'cold_item_rate': 0.0}
    row = relevance.agg(
        F.avg(F.col('is_warm_user').cast('double')).alias('warm_user_rate'),
        F.avg(F.col('is_warm_item').cast('double')).alias('warm_item_rate'),
        F.countDistinct('user_id').alias('users'),
        F.countDistinct('video_id').alias('items'),
    ).first()
    return {
        'rows': int(total), 'users': int(row['users']), 'items': int(row['items']),
        'warm_user_rate': float(row['warm_user_rate'] or 0.0),
        'warm_item_rate': float(row['warm_item_rate'] or 0.0),
        'cold_user_rate': float(1.0 - (row['warm_user_rate'] or 0.0)),
        'cold_item_rate': float(1.0 - (row['warm_item_rate'] or 0.0)),
    }

def ranking_metrics(recommendations, relevance, k=10):
    rel = relevance.where(F.col('is_warm_user') & F.col('is_warm_item')).select('user_idx', 'video_idx', 'graded_relevance')
    rel_by_user = rel.groupBy('user_idx').agg(F.countDistinct('video_idx').alias('num_relevant'))
    evaluated_users = rel_by_user.count()
    if evaluated_users == 0:
        return {'evaluated_users': 0, f'recall@{k}': 0.0, f'ndcg@{k}': 0.0, f'hitrate@{k}': 0.0}
    joined = (
        recommendations.where(F.col('rank') <= k)
        .join(rel, ['user_idx', 'video_idx'], 'left')
        .withColumn('is_hit', F.col('graded_relevance').isNotNull().cast('int'))
        .withColumn('gain', F.coalesce(F.col('graded_relevance'), F.lit(0.0)))
        .withColumn('dcg_term', F.col('gain') / F.log2(F.col('rank') + F.lit(1.0)))
    )
    per_user_dcg = joined.groupBy('user_idx').agg(F.sum('is_hit').alias('hits'), F.sum('dcg_term').alias('dcg'), F.countDistinct('video_idx').alias('recommended_items'))
    ideal_window = Window.partitionBy('user_idx').orderBy(F.desc('graded_relevance'), F.asc('video_idx'))
    ideal = (
        rel.withColumn('ideal_rank', F.row_number().over(ideal_window))
        .where(F.col('ideal_rank') <= k)
        .withColumn('idcg_term', F.col('graded_relevance') / F.log2(F.col('ideal_rank') + F.lit(1.0)))
        .groupBy('user_idx').agg(F.sum('idcg_term').alias('idcg'))
    )
    metrics = (
        rel_by_user.join(per_user_dcg, 'user_idx', 'left')
        .join(ideal, 'user_idx', 'left')
        .fillna({'hits': 0.0, 'dcg': 0.0, 'recommended_items': 0.0, 'idcg': 0.0})
        .withColumn('recall', F.col('hits') / F.least(F.col('num_relevant'), F.lit(k)))
        .withColumn('hitrate', (F.col('hits') > 0).cast('double'))
        .withColumn('ndcg', F.when(F.col('idcg') > 0, F.col('dcg') / F.col('idcg')).otherwise(F.lit(0.0)))
    )
    row = metrics.agg(F.count('*').alias('evaluated_users'), F.avg('recall').alias('recall'), F.avg('ndcg').alias('ndcg'), F.avg('hitrate').alias('hitrate'), F.avg('recommended_items').alias('avg_recommended_items')).first()
    return {
        'evaluated_users': int(row['evaluated_users']),
        f'recall@{k}': float(row['recall'] or 0.0),
        f'ndcg@{k}': float(row['ndcg'] or 0.0),
        f'hitrate@{k}': float(row['hitrate'] or 0.0),
        'avg_recommended_items': float(row['avg_recommended_items'] or 0.0),
    }

def popularity_recommendations(train_interactions, users_to_eval, train_history, k, candidate_multiplier=300):
    top_n = max(k * candidate_multiplier, 1000)
    popular = train_interactions.groupBy('video_idx').agg(F.sum('interaction_strength').alias('popularity_score')).orderBy(F.desc('popularity_score'), F.asc('video_idx')).limit(top_n)
    candidates = users_to_eval.select('user_idx').distinct().crossJoin(F.broadcast(popular))
    filtered = candidates.join(train_history.select('user_idx', 'video_idx'), ['user_idx', 'video_idx'], 'left_anti')
    window = Window.partitionBy('user_idx').orderBy(F.desc('popularity_score'), F.asc('video_idx'))
    return filtered.withColumn('rank', F.row_number().over(window)).where(F.col('rank') <= k)

def train_als(train, params):
    als = ALS(
        userCol='user_idx', itemCol='video_idx', ratingCol='interaction_strength',
        implicitPrefs=True, coldStartStrategy='drop', nonnegative=False,
        rank=params.rank, regParam=params.reg_param, alpha=params.alpha,
        maxIter=params.max_iter, seed=42,
    )
    return als.fit(train.select('user_idx', 'video_idx', 'interaction_strength'))

def als_recommendations(model, users_to_eval, train_history, k, over_generate=300):
    raw = model.recommendForUserSubset(users_to_eval.select('user_idx').distinct(), max(k, over_generate))
    exploded = raw.select('user_idx', F.posexplode('recommendations').alias('pos', 'rec')).select('user_idx', F.col('rec.video_idx').alias('video_idx'), F.col('rec.rating').alias('score'))
    filtered = exploded.join(train_history.select('user_idx', 'video_idx'), ['user_idx', 'video_idx'], 'left_anti')
    window = Window.partitionBy('user_idx').orderBy(F.desc('score'), F.asc('video_idx'))
    return filtered.withColumn('rank', F.row_number().over(window)).where(F.col('rank') <= k)

def log_metrics(prefix, metrics):
    for key, value in metrics.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric(f'{prefix}_{key.replace("@", "_at_")}', float(value))


## Build Gold Dataset

Set `STRENGTH_TUNING_TRIALS > 0` only after the default run works. Direct ALS-per-trial tuning is expensive; this notebook keeps the first run stable.


In [ ]:
K = 10
STRENGTH_TUNING_TRIALS = 0
WEIGHTS = DEFAULT_WEIGHTS

raw = read_interactions(LOCAL_INTERACTIONS)
split, split_meta = split_events(raw)
split = split.cache()
weighted = add_event_strength(split, WEIGHTS).cache()

train_events = weighted.where(F.col('split') == 'train').cache()
train_original = build_train_interactions(train_events).cache()
user_mapping, item_mapping = build_mappings(train_original)
user_mapping = user_mapping.cache()
item_mapping = item_mapping.cache()
train_interactions = apply_mappings(train_original, user_mapping, item_mapping).cache()

validation_relevance = build_relevance(weighted, user_mapping, item_mapping, 'validation', WEIGHTS).cache()
test_relevance = build_relevance(weighted, user_mapping, item_mapping, 'test', WEIGHTS).cache()

train_validation_events = weighted.where(F.col('split').isin('train', 'validation')).cache()
train_validation_original = build_train_interactions(train_validation_events).cache()
final_user_mapping, final_item_mapping = build_mappings(train_validation_original)
final_user_mapping = final_user_mapping.cache()
final_item_mapping = final_item_mapping.cache()
train_validation_interactions = apply_mappings(train_validation_original, final_user_mapping, final_item_mapping).cache()
final_test_relevance = build_relevance(weighted, final_user_mapping, final_item_mapping, 'test', WEIGHTS).cache()

LOCAL_GOLD_DIR.mkdir(parents=True, exist_ok=True)
write_parquet(train_interactions.select('user_idx','video_idx','interaction_strength','user_id','video_id'), LOCAL_GOLD_DIR / 'train_interactions')
write_parquet(validation_relevance, LOCAL_GOLD_DIR / 'validation_relevance')
write_parquet(test_relevance, LOCAL_GOLD_DIR / 'test_relevance')
write_parquet(user_mapping, LOCAL_GOLD_DIR / 'user_mapping')
write_parquet(item_mapping, LOCAL_GOLD_DIR / 'item_mapping')
write_parquet(train_validation_interactions.select('user_idx','video_idx','interaction_strength','user_id','video_id'), LOCAL_GOLD_DIR / 'train_validation_interactions')
write_parquet(final_user_mapping, LOCAL_GOLD_DIR / 'final_user_mapping')
write_parquet(final_item_mapping, LOCAL_GOLD_DIR / 'final_item_mapping')
write_parquet(final_test_relevance, LOCAL_GOLD_DIR / 'final_test_relevance')

manifest = {
    'gold_dataset_version': 'als_v1_colab',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_drive_interactions': str(DRIVE_INTERACTIONS),
    'output_drive_path': str(DRIVE_GOLD_DIR),
    'interaction_strength': {
        'formula_version': FORMULA_VERSION,
        'formula': 'interaction_strength = log1p(sum(0.5*clip(watch_ratio,0,3)+1.0*long_view+1.5*like+1.5*comment+1.5*forward+2.0*follow))',
        'weights': WEIGHTS,
        'strength_tuning_trials': STRENGTH_TUNING_TRIALS,
        'negative_feedback_policy': 'is_hate is retained but not encoded as negative ALS rating in v1',
    },
    'temporal_split': split_meta,
    'split_summary': split_summary(split),
    'mapping_statistics_train_only': {'users': user_mapping.count(), 'items': item_mapping.count(), 'train_user_item_rows': train_interactions.count()},
    'mapping_statistics_train_validation': {'users': final_user_mapping.count(), 'items': final_item_mapping.count(), 'train_validation_user_item_rows': train_validation_interactions.count()},
    'cold_start': {
        'validation': cold_start_report(validation_relevance),
        'test': cold_start_report(test_relevance),
        'final_test_after_train_validation': cold_start_report(final_test_relevance),
    },
    'evaluation_protocol': {
        'split': 'event-level chronological split before user-item aggregation',
        'k': K,
        'train_history_filter': 'recommendations exclude items seen in training history',
        'metrics': [f'ndcg@{K}', f'recall@{K}', f'hitrate@{K}'],
        'relevance': 'binary relevant if any future long_view/like/comment/forward/follow; graded relevance from positive engagement weights',
    },
}
write_json(manifest, LOCAL_GOLD_DIR / 'manifest.json')
print(json.dumps(manifest, indent=2)[:5000])


## Train Popularity + ALS Baselines

Default Colab grid below is stronger than local-safe but still moderate. If runtime/disk fails, reduce to:

```python
RANKS = [16, 32]
MAX_ITER = 3
ALS_OVER_GENERATE = 200
```


In [ ]:
RANKS = [32, 64]
REG_PARAMS = [0.05, 0.1]
ALPHAS = [10.0, 20.0]
MAX_ITER = 5
ALS_OVER_GENERATE = 300
MAX_GRID_MODELS = 0  # 0 = all combinations
RUN_NAME = 'als_baseline_v1_colab'


In [ ]:
if MLFLOW_AVAILABLE:
    LOCAL_MLRUNS.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(LOCAL_MLRUNS.resolve().as_uri())
    mlflow.set_experiment('kuairand_als_baseline_colab')
else:
    print('MLflow logging disabled. Metrics will still be written to evaluation_summary.json.')

train = spark.read.parquet(str(LOCAL_GOLD_DIR / 'train_interactions')).cache()
validation_relevance = spark.read.parquet(str(LOCAL_GOLD_DIR / 'validation_relevance')).cache()
test_relevance = spark.read.parquet(str(LOCAL_GOLD_DIR / 'test_relevance')).cache()
train_validation = spark.read.parquet(str(LOCAL_GOLD_DIR / 'train_validation_interactions')).cache()
final_test_relevance = spark.read.parquet(str(LOCAL_GOLD_DIR / 'final_test_relevance')).cache()

train_history = train.select('user_idx', 'video_idx').cache()
train_validation_history = train_validation.select('user_idx', 'video_idx').cache()
validation_users = validation_relevance.where(F.col('is_warm_user')).select('user_idx').distinct()
test_users = test_relevance.where(F.col('is_warm_user')).select('user_idx').distinct()
final_test_users = final_test_relevance.where(F.col('is_warm_user')).select('user_idx').distinct()

grid = [AlsParams(rank=r, reg_param=reg, alpha=a, max_iter=MAX_ITER) for r, reg, a in itertools.product(RANKS, REG_PARAMS, ALPHAS)]
if MAX_GRID_MODELS:
    grid = grid[:MAX_GRID_MODELS]
print('Grid size:', len(grid), [asdict(p) for p in grid])

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({'k': K, 'max_iter': MAX_ITER, 'als_over_generate': ALS_OVER_GENERATE, 'grid_size': len(grid), 'formula_version': FORMULA_VERSION})
    for name, value in WEIGHTS.items():
        mlflow.log_param(f'strength_weight_{name}', value)

    pop_val_recs = popularity_recommendations(train, validation_users, train_history, K).cache()
    pop_test_recs = popularity_recommendations(train, test_users, train_history, K).cache()
    pop_validation = ranking_metrics(pop_val_recs, validation_relevance, K)
    pop_test = ranking_metrics(pop_test_recs, test_relevance, K)
    log_metrics('popularity_validation', pop_validation)
    log_metrics('popularity_test', pop_test)

    best = None
    als_results = []
    for idx, params in enumerate(grid):
        with mlflow.start_run(run_name=f'als_grid_{idx}', nested=True):
            mlflow.log_params(asdict(params))
            model = train_als(train, params)
            recs = als_recommendations(model, validation_users, train_history, K, over_generate=ALS_OVER_GENERATE).cache()
            metrics = ranking_metrics(recs, validation_relevance, K)
            log_metrics('validation', metrics)
            impact = metrics[f'ndcg@{K}'] - pop_validation[f'ndcg@{K}']
            mlflow.log_metric('validation_impact_ndcg_vs_popularity', impact)
            result = {'params': asdict(params), 'validation': metrics}
            als_results.append(result)
            score = metrics[f'ndcg@{K}']
            if best is None or score > best[0]:
                best = (score, params)
            recs.unpersist()
            print('Finished', result)

    selected = best[1]
    mlflow.log_params({f'selected_{k}': v for k, v in asdict(selected).items()})
    final_model = train_als(train_validation, selected)
    final_recs = als_recommendations(final_model, final_test_users, train_validation_history, K, over_generate=ALS_OVER_GENERATE).cache()
    final_test = ranking_metrics(final_recs, final_test_relevance, K)
    log_metrics('final_als_test', final_test)

    impact = {
        f'ndcg@{K}': final_test[f'ndcg@{K}'] - pop_test[f'ndcg@{K}'],
        f'recall@{K}': final_test[f'recall@{K}'] - pop_test[f'recall@{K}'],
        f'hitrate@{K}': final_test[f'hitrate@{K}'] - pop_test[f'hitrate@{K}'],
    }
    for key, value in impact.items():
        mlflow.log_metric(f'final_test_impact_{key.replace("@", "_at_")}_vs_popularity', value)

    write_parquet(final_model.userFactors.withColumnRenamed('id', 'user_idx'), LOCAL_GOLD_DIR / 'user_factors')
    write_parquet(final_model.itemFactors.withColumnRenamed('id', 'video_idx'), LOCAL_GOLD_DIR / 'item_factors')

    evaluation = {
        'mlflow_run_id': run.info.run_id,
        'mlflow_tracking_uri': mlflow.get_tracking_uri(),
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'popularity_validation': pop_validation,
        'popularity_test': pop_test,
        'als_grid_results': als_results,
        'selected_als_hyperparameters': asdict(selected),
        'final_als_test': final_test,
        'impact_vs_popularity_test': impact,
    }
    manifest.update({
        'trained_at': evaluation['generated_at'],
        'mlflow': {'tracking_uri': mlflow.get_tracking_uri(), 'experiment_name': 'kuairand_als_baseline_colab', 'run_id': run.info.run_id},
        'popularity_baseline': {'validation': pop_validation, 'test': pop_test},
        'als_grid_results': als_results,
        'selected_als_hyperparameters': asdict(selected),
        'final_als_test_metrics': final_test,
        'impact_vs_popularity_test': impact,
        'artifact_paths': {'user_factors': str(LOCAL_GOLD_DIR / 'user_factors'), 'item_factors': str(LOCAL_GOLD_DIR / 'item_factors')},
    })
    write_json(evaluation, LOCAL_GOLD_DIR / 'evaluation_summary.json')
    write_json(manifest, LOCAL_GOLD_DIR / 'manifest.json')
    mlflow.log_artifact(str(LOCAL_GOLD_DIR / 'manifest.json'))
    mlflow.log_artifact(str(LOCAL_GOLD_DIR / 'evaluation_summary.json'))

print(json.dumps(evaluation, indent=2))


## Verify Saved Factors


In [ ]:
user_factors = spark.read.parquet(str(LOCAL_GOLD_DIR / 'user_factors'))
item_factors = spark.read.parquet(str(LOCAL_GOLD_DIR / 'item_factors'))
print('user_factors', user_factors.count(), user_factors.schema.simpleString())
print('item_factors', item_factors.count(), item_factors.schema.simpleString())


## Copy Artifacts Back to Drive

This preserves outputs after the Colab runtime shuts down.


In [ ]:
# Remove old Drive outputs and copy fresh local artifacts back.
if DRIVE_GOLD_DIR.exists():
    shutil.rmtree(DRIVE_GOLD_DIR)
DRIVE_GOLD_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_GOLD_DIR, DRIVE_GOLD_DIR)

if LOCAL_MLRUNS.exists():
    if DRIVE_MLRUNS.exists():
        shutil.rmtree(DRIVE_MLRUNS)
    DRIVE_MLRUNS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(LOCAL_MLRUNS, DRIVE_MLRUNS)
    print('Copied mlruns to', DRIVE_MLRUNS)
else:
    print('No MLflow directory found; skipped copying mlruns.')

print('Copied gold to', DRIVE_GOLD_DIR)


## Optional: Inspect MLflow Later

After downloading/syncing `MyDrive/recsys/data/mlruns`, you can run locally:

```bash
mlflow ui --backend-store-uri data/mlruns
```
